<a href="https://colab.research.google.com/github/rhigazi/prepareTraining/blob/main/TurningPoint_Training_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 15-Minuten Wendepunkt-Erkennung (Turning-Point Classifier) – Google Colab Training Notebook

Dieses Notebook setzt das Trainingskonzept auf Basis der mit `prepareTraining.py` erzeugten Trainings-CSVs um:
- **Tickerunabhängige Features (`feat_*`)**: Es werden ausschließlich skalierungsfreie, relative Merkmale verwendet (keine rohen Kurse aus `meta_*`).
- **Kein Look-Ahead-Leakage**: Chronologischer Split pro Ticker (`Train` / `Validation` / `Test`) sowie optionaler **Out-of-Ticker-Test** für komplett ungesehene Ticker.
- **Umgang mit Klassenungleichgewicht**: Gewichtete Cross-Entropy-Loss-Funktion (`class_weights`) + optionale Gewichtung über `label_max_move_atr`.
- **Saubere Auswertung**: Precision, Recall, F1 pro Klasse (`-1` Extrem-Hoch, `0` Neutral, `+1` Extrem-Tief), Konfusionsmatrix, Schwellenwert-Tuning und Chart-Visualisierung der erkannten Wendepunkte.

In [ ]:
# 1. Setup & Imports
import os
import glob
import random
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Verwendetes Gerät:", device)

In [ ]:
# 2. Konfiguration (Pfade, Split & Hyperparameter)
# Lege eine oder mehrere mit prepareTraining.py erzeugte CSV-Dateien ab (z. B. in /content/training_data/*.csv)
DATA_DIR = "/content/training_data"
SINGLE_FALLBACK_CSV = "/content/output (1).csv"  # Falls nur eine einzelne Datei direkt unter /content liegt

# Optional: Liste von Ticker-Dateinamen (ohne .csv) für den strengen Out-of-Ticker-Härtetest
OUT_OF_TICKER_NAMES = []  # z. B. ["NVDA_train", "SPY_train"]

# Chronologischer Split pro Ticker
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
# Rest (0.15) = Test

# Modell- & Trainingsparameter
BATCH_SIZE = 64
EPOCHS = 60
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
DROPOUT = 0.25
HIDDEN_DIMS = [64, 32]
USE_MOVE_ATR_WEIGHTING = True   # Stärkere Swings (label_max_move_atr) im Loss leicht höher gewichten
PATIENCE = 10                   # Early Stopping auf Validation Macro-F1 (für -1 und +1)

In [ ]:
# 3. Daten laden (Colab Upload oder Ordner / Google Drive)
# Optional für Google Drive:
# from google.colab import drive
# drive.mount('/content/drive')

os.makedirs(DATA_DIR, exist_ok=True)

csv_files = sorted(glob.glob(os.path.join(DATA_DIR, "*.csv")))
if not csv_files and os.path.exists(SINGLE_FALLBACK_CSV):
    csv_files = [SINGLE_FALLBACK_CSV]

if not csv_files:
    try:
        from google.colab import files
        print("Bitte lade eine oder mehrere Trainings-CSVs hoch (erzeugt mit prepareTraining.py):")
        uploaded = files.upload()
        for fname in uploaded.keys():
            dst = os.path.join(DATA_DIR, fname)
            os.replace(fname, dst)
        csv_files = sorted(glob.glob(os.path.join(DATA_DIR, "*.csv")))
    except ImportError:
        raise FileNotFoundError(f"Keine CSV-Dateien in {DATA_DIR} oder {SINGLE_FALLBACK_CSV} gefunden.")

print(f"Gefundene Trainings-CSVs ({len(csv_files)}):", [os.path.basename(f) for f in csv_files])

In [ ]:
# 4. Feature-Check & chronologischer Split pro Ticker (kein Look-Ahead-Leakage)
# Mapping der Labels: -1 (Extrem-Hoch) -> 0, 0 (Neutral) -> 1, +1 (Extrem-Tief) -> 2
LABEL_TO_IDX = {-1: 0, 0: 1, 1: 2}
IDX_TO_LABEL = {0: -1, 1: 0, 2: 1}
CLASS_NAMES = ["Extrem-Hoch (-1)", "Neutral (0)", "Extrem-Tief (+1)"]

def load_and_split_datasets(file_paths):
    train_dfs, val_dfs, test_dfs, oot_dfs = [], [], [], []
    feature_cols = None

    for path in file_paths:
        ticker_name = os.path.splitext(os.path.basename(path))[0]
        df = pd.read_csv(path)
        df["ticker_source"] = ticker_name

        # Strikte Trennung: NUR feat_* Spalten für das Modell zulassen
        curr_feats = [c for c in df.columns if c.startswith("feat_")]
        if not curr_feats:
            raise ValueError(f"Keine feat_*-Spalten in {path} gefunden.")
        if feature_cols is None:
            feature_cols = curr_feats
        elif curr_feats != feature_cols:
            raise ValueError(f"Abweichende Feature-Spalten in {path}: {curr_feats} vs {feature_cols}")

        required_labels = ["label_turn", "label_max_move_atr"]
        for col in required_labels:
            if col not in df.columns:
                raise ValueError(f"Pflichtspalte {col} fehlt in {path}.")

        # Sicherheitscheck auf NaN / inf
        df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=feature_cols + required_labels).reset_index(drop=True)
        df["target_idx"] = df["label_turn"].astype(int).map(LABEL_TO_IDX)

        if ticker_name in OUT_OF_TICKER_NAMES:
            oot_dfs.append(df)
            continue

        n = len(df)
        train_end = int(n * TRAIN_RATIO)
        val_end = int(n * (TRAIN_RATIO + VAL_RATIO))

        train_dfs.append(df.iloc[:train_end].copy())
        val_dfs.append(df.iloc[train_end:val_end].copy())
        test_dfs.append(df.iloc[val_end:].copy())

    df_train = pd.concat(train_dfs, ignore_index=True)
    df_val = pd.concat(val_dfs, ignore_index=True)
    df_test = pd.concat(test_dfs, ignore_index=True)
    df_oot = pd.concat(oot_dfs, ignore_index=True) if oot_dfs else pd.DataFrame(columns=df_train.columns)

    return df_train, df_val, df_test, df_oot, feature_cols

df_train, df_val, df_test, df_oot, FEATURE_COLS = load_and_split_datasets(csv_files)

print(f"Anzahl Features ({len(FEATURE_COLS)}):", FEATURE_COLS)
print(f"Split-Größen -> Train: {len(df_train)} | Val: {len(df_val)} | Test: {len(df_test)} | Out-of-Ticker: {len(df_oot)}")
print("\nKlassenverteilung im Training:")
print(df_train["label_turn"].value_counts().sort_index())

In [ ]:
# 5. Feature-Skalierung (nur auf Train fitten!) & PyTorch DataLoader
scaler = StandardScaler()
X_train = scaler.fit_transform(df_train[FEATURE_COLS].to_numpy(dtype=np.float32))
X_val = scaler.transform(df_val[FEATURE_COLS].to_numpy(dtype=np.float32))
X_test = scaler.transform(df_test[FEATURE_COLS].to_numpy(dtype=np.float32))

y_train = df_train["target_idx"].to_numpy(dtype=np.int64)
y_val = df_val["target_idx"].to_numpy(dtype=np.int64)
y_test = df_test["target_idx"].to_numpy(dtype=np.int64)

# Sample-Gewichte optional über label_max_move_atr für Wendepunkte
w_train = np.ones(len(df_train), dtype=np.float32)
if USE_MOVE_ATR_WEIGHTING:
    turn_mask = df_train["label_turn"].to_numpy() != 0
    move_atr = df_train["label_max_move_atr"].to_numpy(dtype=np.float32)
    # Begrenzte Gewichtung zwischen 1.0 und 2.5 für ausgeprägte Wendepunkte
    w_train[turn_mask] = np.clip(move_atr[turn_mask] / 1.5, 1.0, 2.5)

train_ds = TensorDataset(torch.tensor(X_train), torch.tensor(y_train), torch.tensor(w_train))
val_ds = TensorDataset(torch.tensor(X_val), torch.tensor(y_val))
test_ds = TensorDataset(torch.tensor(X_test), torch.tensor(y_test))

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE * 2, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE * 2, shuffle=False)

# Klassen-Gewichte gegen das Ungleichgewicht (~91.6 % Neutral vs. je ~4.2 % Turns)
class_counts = np.bincount(y_train, minlength=3)
raw_weights = len(y_train) / (3.0 * np.maximum(class_counts, 1))
# Sanfte Dämpfung (Wurzel), damit nicht zu viele Fehlalarme entstehen
class_weights = np.sqrt(raw_weights)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32, device=device)

print("Klassen-Zählung (Train):", dict(zip(CLASS_NAMES, class_counts)))
print("Verwendete Loss-Klassengewichte:", np.round(class_weights, 3))

In [ ]:
# 6. Kompakter Turning-Point-Klassifikator (PyTorch MLP)
class TurningPointMLP(nn.Module):
    def __init__(self, in_features: int, hidden_dims: list, dropout: float = 0.25, num_classes: int = 3):
        super().__init__()
        layers = []
        prev_dim = in_features
        for h_dim in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, h_dim),
                nn.BatchNorm1d(h_dim),
                nn.SiLU(),
                nn.Dropout(dropout),
            ])
            prev_dim = h_dim
        layers.append(nn.Linear(prev_dim, num_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

model = TurningPointMLP(
    in_features=len(FEATURE_COLS),
    hidden_dims=HIDDEN_DIMS,
    dropout=DROPOUT,
    num_classes=3
).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights_tensor, reduction="none")
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=4)

print(model)

In [ ]:
# 7. Training mit Validation-Überwachung (Fokus auf Turn-F1 für -1 und +1)
def predict_proba(model, loader):
    model.eval()
    all_probs, all_targets = [], []
    with torch.no_grad():
        for batch in loader:
            xb, yb = batch[0].to(device), batch[1]
            logits = model(xb)
            probs = torch.softmax(logits, dim=-1).cpu().numpy()
            all_probs.append(probs)
            all_targets.append(yb.numpy())
    return np.vstack(all_probs), np.concatenate(all_targets)

history = {"train_loss": [], "val_turn_f1": [], "val_turn_prec": [], "val_turn_rec": []}
best_val_f1 = -1.0
best_state = None
epochs_no_improve = 0

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    total_samples = 0

    for xb, yb, wb in train_loader:
        xb, yb, wb = xb.to(device), yb.to(device), wb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        per_sample_loss = criterion(logits, yb)
        loss = (per_sample_loss * wb).mean()
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * len(yb)
        total_samples += len(yb)

    train_loss = running_loss / max(total_samples, 1)

    val_probs, val_true = predict_proba(model, val_loader)
    val_pred = val_probs.argmax(axis=1)

    # Metriken gezielt für Klasse 0 (-1 Extrem-Hoch) und Klasse 2 (+1 Extrem-Tief)
    prec, rec, f1, _ = precision_recall_fscore_support(
        val_true, val_pred, labels=[0, 2], average="macro", zero_division=0
    )

    scheduler.step(f1)
    history["train_loss"].append(train_loss)
    history["val_turn_f1"].append(f1)
    history["val_turn_prec"].append(prec)
    history["val_turn_rec"].append(rec)

    if f1 > best_val_f1:
        best_val_f1 = f1
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoche {epoch:02d}/{EPOCHS} | Train Loss: {train_loss:.4f} | "
              f"Val Turn-Precision: {prec:.3f} | Val Turn-Recall: {rec:.3f} | Val Turn-F1: {f1:.3f}")

    if epochs_no_improve >= PATIENCE:
        print(f"Early Stopping in Epoche {epoch} (bester Val Turn-F1: {best_val_f1:.3f}).")
        break

if best_state is not None:
    model.load_state_dict(best_state)

In [ ]:
# 8. Lernkurve & Auswertung auf Test- sowie Out-of-Ticker-Daten
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history["train_loss"], label="Train Loss")
axes[0].set_title("Trainings-Loss")
axes[0].set_xlabel("Epoche")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].plot(history["val_turn_f1"], label="Val Turn-F1 (-1 & +1)")
axes[1].plot(history["val_turn_prec"], label="Val Turn-Precision", linestyle="--")
axes[1].plot(history["val_turn_rec"], label="Val Turn-Recall", linestyle=":")
axes[1].set_title("Validation Wendepunkt-Metriken")
axes[1].set_xlabel("Epoche")
axes[1].grid(True, alpha=0.3)
axes[1].legend()
plt.tight_layout()
plt.show()

def evaluate_split(name, loader):
    probs, y_true = predict_proba(model, loader)
    y_pred = probs.argmax(axis=1)
    print(f"=== Auswertung: {name} ===")
    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=3, zero_division=0))
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])
    cm_df = pd.DataFrame(cm, index=[f"True: {c}" for c in CLASS_NAMES], columns=[f"Pred: {c}" for c in CLASS_NAMES])
    print("Konfusionsmatrix:\n", cm_df, "\n")
    return probs, y_true, y_pred

test_probs, test_true, test_pred = evaluate_split("Test-Set (Out-of-Sample)", test_loader)

if len(df_oot) > 0:
    X_oot = scaler.transform(df_oot[FEATURE_COLS].to_numpy(dtype=np.float32))
    y_oot = df_oot["target_idx"].to_numpy(dtype=np.int64)
    oot_loader = DataLoader(TensorDataset(torch.tensor(X_oot), torch.tensor(y_oot)), batch_size=BATCH_SIZE * 2)
    evaluate_split("Out-of-Ticker-Set (komplett ungesehene Ticker)", oot_loader)

In [ ]:
# 9. Konfidenz-Schwellenwert prüfen (Fehlalarme reduzieren)
# Nur wenn P(Extrem-Hoch) oder P(Extrem-Tief) >= THRESHOLD liegt, wird ein Wendepunkt markiert
val_probs, val_true = predict_proba(model, val_loader)

print("Schwellenwert-Analyse auf Validation-Daten:")
for thresh in [0.35, 0.45, 0.55, 0.65, 0.75]:
    pred_idx = np.ones(len(val_probs), dtype=int)  # Standard: 1 = Neutral (0)
    high_conf = val_probs[:, 0] >= thresh
    low_conf = val_probs[:, 2] >= thresh
    # Falls beide über Schwelle, nimm höhere Wahrscheinlichkeit
    both = high_conf & low_conf
    high_conf = (high_conf & ~both) | (both & (val_probs[:, 0] > val_probs[:, 2]))
    low_conf = (low_conf & ~both) | (both & (val_probs[:, 2] > val_probs[:, 0]))

    pred_idx[high_conf] = 0
    pred_idx[low_conf] = 2

    prec, rec, f1, _ = precision_recall_fscore_support(
        val_true, pred_idx, labels=[0, 2], average="macro", zero_division=0
    )
    n_signals = int((pred_idx != 1).sum())
    print(f"Threshold {thresh:.2f} -> Signale: {n_signals:4d} | Turn-Precision: {prec:.3f} | Turn-Recall: {rec:.3f} | Turn-F1: {f1:.3f}")

In [ ]:
# 10. Visueller Chart-Check der erkannten Extrem- und Wendepunkte auf dem Test-Set
CONFIDENCE_THRESHOLD = 0.50
PLOT_BARS = 200  # Anzahl der letzten Test-Kerzen für die Ansicht

plot_df = df_test.tail(PLOT_BARS).copy().reset_index(drop=True)
plot_probs = test_probs[-len(plot_df):]

pred_turn = np.zeros(len(plot_df), dtype=int)
for i in range(len(plot_df)):
    p_high, _, p_low = plot_probs[i]
    if p_high >= CONFIDENCE_THRESHOLD and p_high > p_low:
        pred_turn[i] = -1
    elif p_low >= CONFIDENCE_THRESHOLD and p_low > p_high:
        pred_turn[i] = 1

plot_df["pred_turn"] = pred_turn

fig, ax = plt.subplots(figsize=(15, 6))
x = np.arange(len(plot_df))
ax.plot(x, plot_df["meta_close"], color="#334155", linewidth=1.2, label="Close (15m)")

# Echte Labels (Ground Truth aus prepareTraining.py)
true_highs = plot_df[plot_df["label_turn"] == -1]
true_lows = plot_df[plot_df["label_turn"] == 1]
ax.scatter(true_highs.index, true_highs["meta_high"] * 1.001, marker="o", facecolors="none",
           edgecolors="#dc2626", s=90, linewidths=1.5, label="Label: Extrem-Hoch (-1)")
ax.scatter(true_lows.index, true_lows["meta_low"] * 0.999, marker="o", facecolors="none",
           edgecolors="#16a34a", s=90, linewidths=1.5, label="Label: Extrem-Tief (+1)")

# Modell-Vorhersagen
pred_highs = plot_df[plot_df["pred_turn"] == -1]
pred_lows = plot_df[plot_df["pred_turn"] == 1]
ax.scatter(pred_highs.index, pred_highs["meta_high"] * 1.0025, marker="v", color="#dc2626",
           s=70, label=f"Modell: Hoch (P >= {CONFIDENCE_THRESHOLD})")
ax.scatter(pred_lows.index, pred_lows["meta_low"] * 0.9975, marker="^", color="#16a34a",
           s=70, label=f"Modell: Tief (P >= {CONFIDENCE_THRESHOLD})")

ax.set_title("15-Minuten Test-Chart: Wahre Wendepunkte (Kreise) vs. Modell-Markierungen (Dreiecke)")
ax.set_xlabel("Kerzen-Index im Testfenster")
ax.set_ylabel("Kurs (nur zur Visualisierung)")
ax.grid(True, alpha=0.25)
ax.legend(loc="best")
plt.tight_layout()
plt.show()

In [ ]:
# 11. Trainiertes Modell & Scaler für spätere Inferenz / RL-Umgebung speichern
ARTIFACT_DIR = "/content/turning_point_artifacts"
os.makedirs(ARTIFACT_DIR, exist_ok=True)

torch.save({
    "model_state_dict": model.state_dict(),
    "feature_cols": FEATURE_COLS,
    "hidden_dims": HIDDEN_DIMS,
    "dropout": DROPOUT,
    "label_to_idx": LABEL_TO_IDX,
    "idx_to_label": IDX_TO_LABEL,
}, os.path.join(ARTIFACT_DIR, "turning_point_mlp.pt"))

joblib.dump(scaler, os.path.join(ARTIFACT_DIR, "feature_scaler.joblib"))
print("Artefakte gespeichert unter:", ARTIFACT_DIR)